In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Button, Label, Dropdown, FloatText, Output, Layout, Accordion
from IPython.display import display, clear_output
import os
import io
import re

# --- Configuration ---
GEOMETRY_FILE = "geometry_fosil.json" 
NEW_GEOMETRY_FILE = "quercus_geometry_edited.json"

# --- STRICT NAME LISTS FOR CATEGORIZATION ---
FRUIT_EXPLICIT_NAMES = [
    'fruitprofile', 'bractaxis', 'bractsection', 'bractwidth', 'infloaxis', 'bractsectionprofile', 'fruitgrowth'
]
LEAF_CORE_NAMES = [
    'leafsection', 'leafwidthgrowth', 'leafwidth', 'petiolecurve'
]
# --------------------------------------------------------

output_log = Output()
curve_data = {}
point_widgets = []
current_curve_id = None
all_curves = []
original_geometry_data = {}

# --- Utility Functions (Keep as is) ---
def is_curve(item):
    return item.get('type') in ['BezierCurve2D', 'NurbsCurve2D']

def normalize_name(name):
    return name.lower().replace('_', '').replace('-', '')

def extract_all_curves(data):
    curves = []
    parameter_groups = data.get('parameters', [])
    if not parameter_groups:
        with output_log:
            print("WARNING: 'parameters' key not found in JSON data.")
            
    for group in parameter_groups:
        group_name = group.get('name', 'UNKNOWN_GROUP')
        items = group.get('items', [])
        for item in items:
            if is_curve(item):
                curve_name = item.get('name', 'UNKNOWN_CURVE')
                item['__group_name'] = group_name
                item['__curve_path'] = f"{group_name}.{curve_name}"
                
                ctrl_list = item.get('ctrlPointList')
                if ctrl_list and ctrl_list.get('type') == 'Point3Array' and 'data' in ctrl_list:
                    item['ctrl_points_ref'] = ctrl_list['data']
                    curves.append(item)
                else:
                    with output_log:
                        print(f"WARNING: Curve '{curve_name}' in group '{group_name}' is missing 'ctrlPointList' data.")
                        
    return curves

def classify_curve(curve):
    name = normalize_name(curve.get('name', ''))
    
    if name in FRUIT_EXPLICIT_NAMES or 'inflo' in name or 'fruit' in name or 'bract' in name:
        return 'Fruit / Inflorescence'
    
    if name in LEAF_CORE_NAMES or 'leaf' in name or 'petiole' in name:
        return 'Leaf'
        
    if 'axis' in name:
        return 'Leaf'
        
    return 'Other'

def load_geometry_data():
    global all_curves, curve_data, original_geometry_data
    try:
        # Check if the simulated JSON file is missing and create a dummy one if so
        if not os.path.exists(GEOMETRY_FILE):
            with output_log:
                print(f"INFO: Geometry file '{GEOMETRY_FILE}' not found. Generating a dummy JSON...")
            
            # --- Dummy Data Generation for Test ---
            dummy_data = {
                "name": "vmango_geometry",
                "parameters": [
                    {
                        "name": "LeafGroup",
                        "items": [
                            {"name": "leafSection", "type": "BezierCurve2D", "id": 101, 
                             "ctrlPointList": {"type": "Point3Array", "data": [[0.0, 0.0, 0.0], [0.5, 0.1, 0.0], [0.9, 0.2, 0.0], [1.0, 0.0, 0.0]]}},
                            {"name": "leafWidthGrowth", "type": "BezierCurve2D", "id": 102, 
                             "ctrlPointList": {"type": "Point3Array", "data": [[0.0, 0.0, 0.0], [0.2, 0.8, 0.0], [0.8, 1.2, 0.0], [1.0, 1.0, 0.0]]}},
                            {"name": "generalAxis", "type": "BezierCurve2D", "id": 105, 
                             "ctrlPointList": {"type": "Point3Array", "data": [[0.0, 0.0, 0.0], [0.3, 0.1, 0.0], [0.7, 0.1, 0.0], [1.0, 0.0, 0.0]]}},
                        ]
                    },
                    {
                        "name": "FruitGroup",
                        "items": [
                            {"name": "fruitProfile", "type": "BezierCurve2D", "id": 201, 
                             "ctrlPointList": {"type": "Point3Array", "data": [[0.0, 0.0, 0.0], [0.1, 0.6, 0.0], [0.9, 0.6, 0.0], [1.0, 0.0, 0.0]]}},
                            {"name": "bractSection", "type": "BezierCurve2D", "id": 203, 
                             "ctrlPointList": {"type": "Point3Array", "data": [[0.0, 0.0, 0.0], [0.1, 0.1, 0.0], [0.9, 0.1, 0.0], [1.0, 0.0, 0.0]]}},
                            {"name": "bractAxis", "type": "BezierCurve2D", "id": 202, 
                             "ctrlPointList": {"type": "Point3Array", "data": [[0.0, 0.0, 0.0], [0.2, 0.05, 0.0], [0.8, 0.05, 0.0], [1.0, 0.0, 0.0]]}},
                        ]
                    },
                     {
                        "name": "OtherGroup",
                        "items": [
                            {"name": "rootCurveA", "type": "BezierCurve2D", "id": 301, 
                             "ctrlPointList": {"type": "Point3Array", "data": [[0.0, 0.0, 0.0], [0.1, 0.2, 0.0], [0.3, 0.1, 0.0], [0.5, 0.0, 0.0]]}},
                        ]
                    }
                ]
            }
            
            with open(GEOMETRY_FILE, 'w') as f:
                json.dump(dummy_data, f, indent=4)
            original_geometry_data = dummy_data
        else:
            with open(GEOMETRY_FILE, 'r') as f:
                original_geometry_data = json.load(f)

        
        all_curves = extract_all_curves(original_geometry_data)
        
        curve_data = {}
        for curve in all_curves:
            category = classify_curve(curve)
            if category not in curve_data:
                curve_data[category] = []
            curve_data[category].append(curve)
            
        with output_log:
            print(f"SUCCESS: Loaded {len(all_curves)} curves from JSON.")
            print(f"Found groups: {list(curve_data.keys())}")
            
        return original_geometry_data
            
    except Exception as e:
        with output_log:
            print(f"ERROR loading data: {e}")
        return {}

def save_geometry_json(filepath):
    global original_geometry_data
    try:
        with open(filepath, 'w') as f:
            json.dump(original_geometry_data, f, indent=4)
        
        with output_log:
            print(f"INFO: Successfully saved modifications to {filepath}")
            
    except Exception as e:
        with output_log:
            print(f"ERROR saving data: {e}")

# --- Curve Mathematics and Plotting Utilities (Keep as is) ---
def create_bezier_points(ctrl_points, num_points=100):
    ctrl_points = np.array(ctrl_points)
    if ctrl_points.shape[0] != 4:
        return np.array([[cp[0], cp[1]] for cp in ctrl_points])
        
    t = np.linspace(0, 1, num_points)
    points = (
        np.outer((1 - t)**3, ctrl_points[0]) +
        np.outer(3 * (1 - t)**2 * t, ctrl_points[1]) +
        np.outer(3 * (1 - t) * t**2, ctrl_points[2]) +
        np.outer(t**3, ctrl_points[3])
    )
    return points[:, :2]

plot_out = Output()
point_control_box = Output()

def update_plot(curve_index, curve_list):
    global current_curve_id
    
    if not curve_list:
        with output_log:
            print("ERROR: No curves to display for this category.")
        return
        
    curve = curve_list[curve_index]
    current_curve_id = curve['id']
    ctrl_points = curve['ctrl_points_ref']
    
    curve_points = create_bezier_points(ctrl_points)
    
    with plot_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6, 6))
        
        ax.plot(curve_points[:, 0], curve_points[:, 1], label=f"Curve: {curve['name']}", color='blue', linewidth=2)
        
        ctrl_points_array = np.array(ctrl_points)[:, :2]
        ax.scatter(ctrl_points_array[:, 0], ctrl_points_array[:, 1], color='red', s=50, zorder=5, label='Control Points')
        ax.plot(ctrl_points_array[:, 0], ctrl_points_array[:, 1], 'r--', alpha=0.5)

        for i, (x, y) in enumerate(ctrl_points_array):
            ax.annotate(f'P{i}', (x, y), textcoords="offset points", xytext=(5,5), ha='center', fontsize=9)
            
        ax.set_title(f"Editing Curve: {curve['name']} ({curve['type']})")
        ax.set_xlabel("X-coordinate")
        ax.set_ylabel("Y-coordinate")
        ax.legend()
        ax.grid(True)
        ax.axis('equal') 
        plt.show()

def update_point_widgets(curve_index, curve_list):
    global point_widgets, current_curve_id
    
    if not curve_list:
        with output_log:
            print("ERROR: No curves for widget update.")
        return
        
    # Crucial step: Clear existing widgets before creating new ones
    for widget in point_widgets:
        widget.close()
    point_widgets.clear()
    
    curve = curve_list[curve_index]
    ctrl_points_data_ref = curve['ctrl_points_ref']
    current_curve_id = curve['id']
    
    for i, point in enumerate(ctrl_points_data_ref):
        x_val = point[0]
        y_val = point[1]
        
        x_text = FloatText(
            value=x_val,
            description=f'P{i} X:',
            style={'description_width': 'initial'},
            layout=Layout(width='180px')
        )
        
        y_text = FloatText(
            value=y_val,
            description=f'P{i} Y:',
            style={'description_width': 'initial'},
            layout=Layout(width='180px')
        )
        
        def create_handler(point_index, coord_index, curve_list_ref, curve_index_ref):
            def handler(change):
                try:
                    curve_list_ref[curve_index_ref]['ctrl_points_ref'][point_index][coord_index] = change.new
                    update_plot(curve_index_ref, curve_list_ref)
                except Exception as e:
                    with output_log:
                        print(f"Error updating point P{point_index} coord {coord_index}: {e}")
            return handler

        x_text.observe(create_handler(i, 0, curve_list, curve_index), names='value')
        y_text.observe(create_handler(i, 1, curve_list, curve_index), names='value')
        
        point_widgets.append(HBox([x_text, y_text], layout=Layout(width='98%')))

    # Display the new widgets once
    with point_control_box:
        clear_output(wait=True)
        display(VBox(point_widgets, layout=Layout(align_items='flex-start')))

# --- Main Execution Function (Final Clean Output) ---

def run_curve_editor():
    """Initializes and displays the interactive curve editor UI."""
    global original_geometry_data
    
    # 0. INITIAL CLEANUP FIX: Clear global Output widgets to prevent ghost artifacts
    with plot_out:
        clear_output(wait=True)
    with point_control_box:
        clear_output(wait=True)
    with output_log:
        clear_output(wait=True)

    # 1. Load the data first
    original_geometry_data = load_geometry_data()

    if not original_geometry_data or not curve_data:
        display(VBox([Label("Setup Failed - Check Log Output:"), output_log]))
        return
    
    # 2. Setup Accordion based on classified groups
    accordion = Accordion()
    titles = []
    children = []
    
    initial_curve_to_draw = None 
    sorted_categories = sorted(curve_data.keys())
    
    for category in sorted_categories:
        curve_list = curve_data.get(category, [])
        curve_names = [curve['name'] for curve in curve_list]
        
        if not curve_names:
            children.append(Label(f"No curves found in {category}."))
            titles.append(f"{category} (0 curves)")
            continue

        curve_selector = Dropdown(
            options=curve_names,
            value=curve_names[0],
            description='Select Curve:',
            style={'description_width': 'initial'},
            layout=Layout(width='98%')
        )

        def create_dropdown_handler(curve_list_ref):
            def handler(change):
                selected_name = change.new
                try:
                    curve_index = next(i for i, c in enumerate(curve_list_ref) if c['name'] == selected_name)
                except StopIteration:
                    with output_log:
                        print(f"ERROR: Could not find curve '{selected_name}' in list.")
                    return
                
                update_point_widgets(curve_index, curve_list_ref)
                update_plot(curve_index, curve_list_ref)
            return handler

        handler_func = create_dropdown_handler(curve_list)
        curve_selector.observe(handler_func, names='value')
        
        children.append(VBox([curve_selector]))
        titles.append(f"{category} ({len(curve_list)} curves)")
        
        if initial_curve_to_draw is None:
            initial_curve_to_draw = {
                'list': curve_list,
                'index': 0
            }


    accordion.children = children
    for i, title in enumerate(titles):
        accordion.set_title(i, title)
        
    # 4. Setup Save Button
    save_button = Button(
        description=f"💾 Save ALL Changes to: {NEW_GEOMETRY_FILE}", 
        style={'button_color': 'darkgreen', 'font_weight': 'bold', 'text_color': 'white'},
        layout=Layout(width='98%', margin='15px 0 10px 0')
    )
    save_button.on_click(lambda b: save_geometry_json(NEW_GEOMETRY_FILE))
    
    # 5. Initial state: Open the first group
    if sorted_categories:
        accordion.selected_index = 0

    # 6. Assemble the UI layout
    left_panel = VBox([
        Label("1. Select Parameter Group and Curve to Edit:", style={'font_weight': 'bold'}),
        accordion,
        Label("2. Adjust Control Points (P0 is start, P3 is end):", style={'font_weight': 'bold', 'margin': '10px 0 5px 0'}),
        point_control_box, 
        save_button,
        Label("\nLog Output:", style={'font_weight': 'bold'}),
        output_log
    ], layout=Layout(width='40%', padding='15px', border='1px solid #ccc', margin='5px', align_items='stretch'))
    
    right_panel = VBox([
        Label("3. Real-time Curve Visualization (Curve in Blue, Controls in Red):", style={'font_weight': 'bold'}),
        plot_out
    ], layout=Layout(width='58%', padding='15px', border='1px solid #ccc', margin='5px', align_items='stretch'))
    
    print("🌲 Quercus Paleo-Botany Geometric Editor (Widgets Initialized)")
    
    # 7. Display the main widget
    display(HBox([left_panel, right_panel], layout=Layout(width='100%', justify_content='space-between')))

    # 8. POST-DISPLAY INITIALIZATION (Draw first curve once)
    if initial_curve_to_draw:
        idx = initial_curve_to_draw['index']
        lst = initial_curve_to_draw['list']
        
        update_point_widgets(idx, lst)
        update_plot(idx, lst)

# Execute the main function to run the editor
if __name__ == '__main__':
    run_curve_editor()

🌲 Quercus Paleo-Botany Geometric Editor (Widgets Initialized)


u